In [6]:
from sequana import DNA
from sequana import FastA
import pandas as pd
import numpy as np
from sklearn.preprocessing import  StandardScaler
import re
from scipy.signal import find_peaks
import matplotlib.pyplot as plt
from scipy.ndimage import uniform_filter1d
from scipy.signal import savgol_filter
from scipy.signal import argrelextrema
from itertools import combinations

from scipy.signal import peak_widths


In [4]:
def count_homopolymers(seq, min_length=5):
    # Ex : trouve AAAAA ou TTTTT, etc.
    pattern = re.compile(rf"(A{{{min_length},}}|T{{{min_length},}}|C{{{min_length},}}|G{{{min_length},}})")
    return len(pattern.findall(seq.upper()))

def load_fasta(fasta_path, window_size=100):
    f = FastA(fasta_path)
    data = []


    for maseq in f:

        print(maseq.name)
        
        features = []

        s = DNA(maseq.sequence.upper())
        seq = maseq.sequence.upper()
        s.window = window_size

       

        #Homopolymere
        X2 = []
        X3 = []

        
        for i in range(0, len(seq)-2, 1):
            
            window = seq[max(0, i - window_size//2):min(i+window_size//2,len(seq))]
            nb = count_homopolymers(window, min_length=2)
            X2.append(nb)

            nb = count_homopolymers(window, min_length=3)
            X3.append(nb)
        


            
        X2 = X2[5000:-5000]
        X3 = X3[5000:-5000]


 






   

        df= pd.DataFrame({
            'X2':X2,
            'X3':X3
        })

        
        data.append(df)

            
    return data

 
data = load_fasta("../data/Fasta/GCA_000002765.1_ASM276v1_genomic.fna",100)


AL844501.1
AE001362.1
AL844502.1
AL844503.1
AL844504.1
AL844505.1
AL844506.2
AL844507.2
AL844508.1
AE014185.2
AE014186.2
AE014188.3
AL844509.2
AE014187.2


In [10]:

df = pd.read_csv("../data/Centromere_Positions/centromeres _donovani.csv")
centromeres = {
    row['Chromosome']: (row['Centromere_Start'], row['Centromere_End'])
    for _, row in df.iterrows()
}






vecteur_Debut = []
vecteur_Fin = []

vecteur_longeur = []


g = 0


for i in range(0,14):
    temp_ =  data[i]['X3']*data[i]['X2']
    temp_ = temp_[40000:-70000]

    mean = np.mean(temp_)

    peaks, properties = find_peaks(temp_,  prominence=mean*2, distance=20)

    window = 3000
    peak_medians = []

 



        #Moyenne
    peak_means = []
    for peak in peaks:
        start = max(0, peak - window//2)
        end = min(len(temp_), peak + window//2)
        mean_val = np.mean(temp_[start:end])
        peak_medians.append((peak, mean_val))
    
  #  for peak in peaks:
  #      start = max(0, peak - window//2)
  #      end = min(len(temp_), peak + window//2 + 1)
    
   #     neighborhood = temp_[start:end]
   #     if len(neighborhood) > 0:
   #         median_val = np.median(neighborhood)
   #         peak_medians.append((peak, median_val))
    
    # Trouver le pic avec la médiane la plus élevée
    if peak_medians:
        peak_max, max_median = max(peak_medians, key=lambda x: x[1])

        max_index = peak_max


        #Chercher la taille du centromere


        # Appliquer un filtre pour lisser
        temp_X2_3 = uniform_filter1d(temp_, size=1500)
        temp_X2_3 = temp_X2_3[max_index-50000:max_index+25000]
        

        
        # Détection des pics
        peaks, properties = find_peaks(temp_X2_3, prominence=5, distance=200)
        # Trouver le pic avec la plus grande **prominence**
        if len(peaks) > 0:
            prominences = properties['prominences']
            peak_index = np.argmax(prominences)
            peak_max = peaks[peak_index]
        
            # Calcule la largeur du pic à mi-hauteur
            widths_result = peak_widths(temp_X2_3, peaks, rel_height=0.5)
        
            taille = widths_result[0][peak_index]
            debut = widths_result[2][peak_index]
            fin = widths_result[3][peak_index]

            debut = debut+max_index-50000+5000+40000
            fin = fin+max_index-50000+5000+40000
            debut = int(debut)
            fin = int(fin)
            max_index = max_index+5000+40000
            # Affichage



      
            print(f'{i+1} : Commence {debut}    Fini {fin}  Taille : {taille}')
   



            vecteur_Debut.append(debut)
            vecteur_Fin.append(fin)
            vecteur_longeur.append(taille)



result = pd.DataFrame()
result['Chromosome'] = list(range(1, 15))
result['start'] = vecteur_Debut 
result['end'] = vecteur_Fin
result['length'] = vecteur_longeur
result.to_csv(f"../output/estimation/plasmodium_X2_X3_M2.csv", index=False)

1 : Commence 458470    Fini 462002  Taille : 3532.0
2 : Commence 447906    Fini 450341  Taille : 2435.0
3 : Commence 594083    Fini 597351  Taille : 3268.0
4 : Commence 648815    Fini 651770  Taille : 2955.0
5 : Commence 183441    Fini 185988  Taille : 2547.0
6 : Commence 478232    Fini 481312  Taille : 3080.0
7 : Commence 864113    Fini 867488  Taille : 3375.0
8 : Commence 299979    Fini 303191  Taille : 3212.0
9 : Commence 1241953    Fini 1244793  Taille : 2840.0
10 : Commence 492686    Fini 494851  Taille : 2165.0
11 : Commence 831173    Fini 834588  Taille : 3415.0
12 : Commence 1282260    Fini 1285402  Taille : 3142.0
13 : Commence 1168230    Fini 1171254  Taille : 3024.0
14 : Commence 1737613    Fini 1739987  Taille : 2374.0
